In [ ]:
import re
import io
import textwrap

import pypdf
import polars as pl
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch

from hexagonal.files.spec import get_polars_dataframe

FONT_SIZE = 12
ADDRESS_TOPLEFT_CORNER = [328, 688]
LINE_SIZE = 1.4
MAX_WIDTH = 35

In [ ]:
deputes_fi = get_polars_dataframe("data/03_main/assemblee_nationale/deputes_lfi.csv")

In [ ]:
ADRESSE = """
Assemblée nationale
126 Rue de l'Université
75355 Paris 07 SP 
""".strip()

In [ ]:
def overlay_adresse(adresse):
    overlay_buffer = io.BytesIO()
    cv = canvas.Canvas(overlay_buffer, pagesize=A4)

    font_size = FONT_SIZE
    lines = adresse.split("\n")

    if len(lines) >= 6:
        font_size -= 1
    
    cv.setFont(
        "Courier",
        font_size
    )

    for i, line in enumerate(lines):
        cv.drawString(
            ADDRESS_TOPLEFT_CORNER[0],
            ADDRESS_TOPLEFT_CORNER[1] - i * LINE_SIZE * font_size,
            line
        )

    cv.save()
    overlay_buffer.seek(0)
    
    return pypdf.PdfReader(overlay_buffer).get_page(0)

In [ ]:
def generer_page(writer, adresse, pages):
    page1 = writer.add_blank_page(*A4)
    page1.merge_page(pages[0])
    page1.merge_page(overlay_adresse(adresse))

    page2 = writer.add_blank_page(*A4)
    page2.merge_page(pages[1])

# Identifier les pages des PDF

In [ ]:
adresse_re = re.compile(r"(?:Cher|Chère) ([\w -]+),")

In [ ]:
templates = {
    ("Mme", "Templates/001_Courrier_de_JLM_pour_nos_parlementaires_insoumis_femmes_ST (2).pdf"),
    ("M.", "Templates/001_Courrier_de_JLM_pour_nos_parlementaires_insoumis_homme_ST (2).pdf"),
    ("M.", "Templates/001_Courrier_de_JLM_pour_nos_parlementaires_insoumis_bérenger_ST.pdf"),
}

In [ ]:
pages = {}

for genre, f in templates:
    reader = pypdf.PdfReader(f)
    for i in range(0, len(reader.pages), 2):
        m = adresse_re.search(reader.pages[i].extract_text())
        pages[genre, m.group(1)] = reader.pages[i], reader.pages[i+1]

In [ ]:
pages["M.", "Éric"] = pages["M.", "Eric"]
pages["M.", "Carlos Martens"] = pages["M.", "Carlos"]

# Associer députés et courriers

In [ ]:
deputes_courriers = deputes_fi.with_columns(
    adresse=(pl.format("{civilite} {prenom} {nom}\n") + ADRESSE).str.to_uppercase()
).sort(["nom", "prenom"])

In [ ]:
writer = pypdf.PdfWriter()
for civilite, prenom, adresse in deputes_courriers.select("civilite", "prenom", "adresse").rows():
    generer_page(writer, adresse, pages[civilite, prenom])

writer.write("out/courrier_deputes.pdf")